<a href="https://colab.research.google.com/github/panarisivalentino-pixel/prediccion-rendimiento-agricola-argentina/blob/main/notebooks/06_dashboard_graficos_agro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dashboard Agrícola Argentina (1981–2024)

Visión general de la producción agrícola argentina: superficie sembrada/cosechada,
producción, rendimiento y variables climáticas asociadas.



In [ ]:


!pip install plotly -q


In [ ]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)


## 1. Carga de datos



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

nombre_buscado = "df_recortado.csv"
ruta_encontrada = None

# Buscar el archivo en Google Drive
for raiz, directorios, archivos in os.walk('/content/drive/MyDrive'):
    if nombre_buscado.lower() in [f.lower() for f in archivos]:
        for archivo in archivos:
            if archivo.lower() == nombre_buscado.lower():
                ruta_encontrada = os.path.join(raiz, archivo)
                break

if ruta_encontrada:
    print(f" ARCHIVO ENCONTRADO EN:\n{ruta_encontrada}")
else:
    print(" No se encontró el archivo con ese nombre exacto. Revisa el paso 3.")

 ARCHIVO ENCONTRADO EN:
/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_recortado.csv


In [ ]:
import pandas as pd


ruta = '/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_recortado.csv'

# Cargar el archivo Excel
df = pd.read_csv(ruta)

# Visualizar las primeras filas
display(df.shape)
df.head()

(111417, 17)

,cultivo,año,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha,T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,precip_total_mm,meses_con_dato
0,ajo,1981,Buenos Aires,6,Bahía Blanca,6056,10,10,39,3900,15.38,28.65,2.54,0.0,71.83,805.9,12.0
1,ajo,1981,Buenos Aires,6,Balcarce,6063,275,275,990,3600,15.38,28.65,2.54,0.0,71.83,805.9,12.0
2,ajo,1981,Buenos Aires,6,Coronel de Marina Leonardo Rosales,6182,10,10,38,3800,15.38,28.65,2.54,0.0,71.83,805.9,12.0
3,ajo,1981,Buenos Aires,6,Coronel Pringles,6196,4,4,18,4500,15.38,28.65,2.54,0.0,71.83,805.9,12.0
4,ajo,1981,Buenos Aires,6,General Pueyrredón,6357,3,3,10,3333,15.38,28.65,2.54,0.0,71.83,805.9,12.0


## 2. Panorama general del dataset

In [ ]:

print("Rango de años:", df['año'].min(), "-", df['año'].max())
print("Cultivos:", df['cultivo'].nunique())
print("Provincias:", df['provincia'].nunique())
print("Departamentos:", df['departamento'].nunique())
print("\nValores nulos por columna:")
print(df.isnull().sum().sum(), "nulos en total")
df.describe()


Rango de años: 1981 - 2024
Cultivos: 41
Provincias: 23
Departamentos: 397

Valores nulos por columna:
0 nulos en total


,año,provincia_id,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,rendimiento_kgxha,T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,precip_total_mm,meses_con_dato
count,111417.000000,111417.000000,111417.000000,111417.000000,111417.000000,1.114170e+05,111417.000000,111417.000000,111417.000000,111417.000000,111417.000000,111417.000000,111417.00000,111417.0
mean,2001.856234,33.045496,33246.989544,15770.584067,14179.062244,4.463207e+04,3609.908362,17.464265,31.422787,4.972014,15.935794,67.120614,989.12029,12.0
std,13.646092,29.481240,29360.978943,41743.990396,40297.629406,1.474816e+05,5248.293918,3.362219,3.757989,3.051941,5.472240,8.590942,376.08904,0.0
min,1981.000000,6.000000,6000.000000,0.000000,0.000000,0.000000e+00,0.000000,5.700000,12.450000,-5.980000,0.000000,33.770000,42.20000,12.0
25%,1989.000000,6.000000,6638.000000,400.000000,200.000000,4.500000e+02,1333.000000,15.000000,28.670000,2.660000,16.900000,61.690000,775.20000,12.0
50%,2002.000000,22.000000,22036.000000,2439.000000,1500.000000,3.500000e+03,2200.000000,16.890000,30.800000,4.670000,17.440000,69.100000,924.40000,12.0
75%,2014.000000,54.000000,54091.000000,12600.000000,9783.000000,2.540400e+04,3700.000000,19.860000,34.040000,7.480000,18.020000,73.140000,1120.30000,12.0
max,2024.000000,94.000000,94000.000000,894613.000000,883613.000000,8.620000e+06,300000.000000,25.810000,40.150000,11.840000,23.020000,86.850000,2649.00000,12.0


In [ ]:

# KPIs principales
total_prod = df['produccion_tm'].sum()
total_sup_sembrada = df['superficie_sembrada_ha'].sum()
total_sup_cosechada = df['superficie_cosechada_ha'].sum()
rendimiento_prom = df['rendimiento_kgxha'].mean()

fig = go.Figure()
kpis = [
    ("Producción total (Tm)", f"{total_prod/1e6:,.1f} M"),
    ("Superficie sembrada (ha)", f"{total_sup_sembrada/1e6:,.1f} M"),
    ("Superficie cosechada (ha)", f"{total_sup_cosechada/1e6:,.1f} M"),
    ("Rendimiento promedio (kg/ha)", f"{rendimiento_prom:,.0f}"),
]
fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}]*4])
for i, (label, value) in enumerate(kpis):
    fig.add_trace(go.Indicator(mode="number", value=None, title={"text": label}), row=1, col=i+1)

# Plotly Indicator necesita valores numéricos; mostramos como tabla simple en su lugar
import plotly.figure_factory as ff
kpi_df = pd.DataFrame(kpis, columns=["Métrica", "Valor"])
fig2 = ff.create_table(kpi_df)
fig2.update_layout(title="📊 KPIs generales del dataset")
fig2.show()


## 3. Evolución temporal (1981–2024)

Cómo evolucionaron la superficie sembrada, la superficie cosechada y la producción
total a lo largo del tiempo, sumando todos los cultivos y provincias.


In [ ]:

evol = df.groupby('año').agg(
    superficie_sembrada_ha=('superficie_sembrada_ha', 'sum'),
    superficie_cosechada_ha=('superficie_cosechada_ha', 'sum'),
    produccion_tm=('produccion_tm', 'sum'),
).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=evol['año'], y=evol['superficie_sembrada_ha'],
                          name='Superficie sembrada (ha)', mode='lines'), secondary_y=False)
fig.add_trace(go.Scatter(x=evol['año'], y=evol['superficie_cosechada_ha'],
                          name='Superficie cosechada (ha)', mode='lines'), secondary_y=False)
fig.add_trace(go.Bar(x=evol['año'], y=evol['produccion_tm'],
                      name='Producción (Tm)', opacity=0.3), secondary_y=True)

fig.update_layout(title="Evolución de superficie y producción total por año",
                   xaxis_title="Año", legend_title="Serie", template="plotly_white")
fig.update_yaxes(title_text="Hectáreas", secondary_y=False)
fig.update_yaxes(title_text="Producción (Tm)", secondary_y=True)
fig.show()


In [ ]:

rend_evol = df.groupby('año')['rendimiento_kgxha'].mean().reset_index()

fig = px.line(rend_evol, x='año', y='rendimiento_kgxha',
              title="Evolución del rendimiento promedio (kg/ha) por año",
              template="plotly_white", markers=True)
fig.update_layout(xaxis_title="Año", yaxis_title="Rendimiento promedio (kg/ha)")
fig.show()


## 4. Ranking de cultivos

In [ ]:

top_cultivos_prod = (df.groupby('cultivo')['produccion_tm'].sum()
                        .sort_values(ascending=False).head(10).reset_index())

fig = px.bar(top_cultivos_prod, x='produccion_tm', y='cultivo', orientation='h',
             title="Top 10 cultivos por producción total acumulada (Tm)",
             template="plotly_white", color='produccion_tm',
             color_continuous_scale='Greens')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title="Producción (Tm)", yaxis_title="")
fig.show()


In [ ]:

top_cultivos_sup = (df.groupby('cultivo')['superficie_sembrada_ha'].sum()
                       .sort_values(ascending=False).head(10).reset_index())

fig = px.bar(top_cultivos_sup, x='superficie_sembrada_ha', y='cultivo', orientation='h',
             title="Top 10 cultivos por superficie sembrada acumulada (ha)",
             template="plotly_white", color='superficie_sembrada_ha',
             color_continuous_scale='Oranges')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title="Superficie sembrada (ha)", yaxis_title="")
fig.show()


In [ ]:

# Rendimiento promedio por cultivo (solo cultivos con suficientes registros)
min_registros = 100
rend_cultivo = (df.groupby('cultivo').filter(lambda g: len(g) >= min_registros)
                   .groupby('cultivo')['rendimiento_kgxha'].mean()
                   .sort_values(ascending=False).head(15).reset_index())

fig = px.bar(rend_cultivo, x='rendimiento_kgxha', y='cultivo', orientation='h',
             title="Top 15 cultivos por rendimiento promedio (kg/ha)",
             template="plotly_white", color='rendimiento_kgxha',
             color_continuous_scale='Blues')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title="Rendimiento promedio (kg/ha)", yaxis_title="")
fig.show()


## 5. Distribución geográfica

In [ ]:

prod_provincia = (df.groupby('provincia')['produccion_tm'].sum()
                     .sort_values(ascending=False).reset_index())

fig = px.bar(prod_provincia, x='provincia', y='produccion_tm',
             title="Producción total acumulada por provincia (Tm)",
             template="plotly_white", color='produccion_tm',
             color_continuous_scale='Viridis')
fig.update_layout(xaxis_title="", yaxis_title="Producción (Tm)", xaxis_tickangle=-45)
fig.show()


In [ ]:

# Mapa de calor: top cultivos x top provincias (superficie sembrada)
top_c = top_cultivos_sup['cultivo'].tolist()
top_p = prod_provincia['provincia'].head(10).tolist()

pivot = (df[df['cultivo'].isin(top_c) & df['provincia'].isin(top_p)]
         .pivot_table(index='cultivo', columns='provincia',
                       values='superficie_sembrada_ha', aggfunc='sum', fill_value=0))
pivot = pivot.loc[top_c, top_p]

fig = px.imshow(pivot, text_auto='.2s', aspect='auto',
                 title="Superficie sembrada (ha): top cultivos x top provincias",
                 color_continuous_scale='YlGnBu')
fig.update_layout(xaxis_title="Provincia", yaxis_title="Cultivo")
fig.show()


## 6. Variables climáticas y su relación con el rendimiento

In [ ]:

clima_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'precip_total_mm']
corr = df[clima_cols + ['rendimiento_kgxha']].corr()

fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                 title="Correlación entre variables climáticas y rendimiento")
fig.show()


In [ ]:

CULTIVO_FILTRO = "soja total"      # ej: "maíz", "trigo total", None para todos
PROVINCIA_FILTRO = None            # ej: "Buenos Aires", None para todas

sub = df.copy()
if CULTIVO_FILTRO:
    sub = sub[sub['cultivo'] == CULTIVO_FILTRO]
if PROVINCIA_FILTRO:
    sub = sub[sub['provincia'] == PROVINCIA_FILTRO]

evol_filtro = sub.groupby('año').agg(
    produccion_tm=('produccion_tm', 'sum'),
    rendimiento_kgxha=('rendimiento_kgxha', 'mean'),
).reset_index()

titulo = f"{CULTIVO_FILTRO or 'Todos los cultivos'} — {PROVINCIA_FILTRO or 'Todas las provincias'}"

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=evol_filtro['año'], y=evol_filtro['produccion_tm'],
                      name='Producción (Tm)', opacity=0.5), secondary_y=False)
fig.add_trace(go.Scatter(x=evol_filtro['año'], y=evol_filtro['rendimiento_kgxha'],
                          name='Rendimiento (kg/ha)', mode='lines+markers'), secondary_y=True)
fig.update_layout(title=titulo, xaxis_title="Año", template="plotly_white")
fig.update_yaxes(title_text="Producción (Tm)", secondary_y=False)
fig.update_yaxes(title_text="Rendimiento (kg/ha)", secondary_y=True)
fig.show()
